# Schizophrenia Pathway Classifier — Classification Modeling

## Objective

In this notebook I am constructing a classification model trained on the expression data to test if the model learns features reflecting the enrichment of immune/inflammatory pathways in differentially expressed genes between SCZ and control within the dataset found in the previous notebook — directly answering the second part of my primary hypothesis. I will engineer features, make preprocessing decisons, and use cross validation to evaluate the models performance. High overlap between the classifier's top features and the Hallmark leading-edge genes from `02_enrichment_analysis.ipynb`'s GSEA (`Lead_genes`) will be evidence in support of the second part of the primary hypothesis, if low or no overlap is found then this test failed to find supporting evidence for this hypothesis.

## Inputs
- `../data/processed/merged_df.csv`
- `../data/processed/gsea_results.csv`
- `../data/processed/hallmark_gene_sets.json`
- `../data/processed/probe_to_gene.json`

## Output
- Features list with per-fold selected gene sets
- Evaluation table with per-sample LOO predictions and metrics
- Feature importance table

## 3.1 Setup & Load Data
Import many of the same libraries from the previous notebooks, with addition to `sklearn` imports for modeling, preprocessing, and evaluating. I set paths and random state for reproducability, and load them into the notebook. 

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from scipy import stats

from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score

RANDOM_STATE = 42
PROCESSED_DATA_PATH = '../data/processed/merged_df.csv'
GSEA_RESULTS_PATH = '../data/processed/gsea_results.csv'
HALLMARK_PATH = '../data/processed/hallmark_gene_sets.json'
PROBE_TO_GENE_PATH = '../data/processed/probe_to_gene.json'

In [18]:
merged_df = pd.read_csv(PROCESSED_DATA_PATH)
# check shape and value counts to confirm df was imported correctly
print(merged_df.shape)
print(merged_df['diagnosis'].value_counts())

gsea_results = pd.read_csv(GSEA_RESULTS_PATH)
# check shape and columns to confirm df was imported correctly
print(gsea_results.shape)
print(gsea_results.columns)

hallmark_set = json.load(open(HALLMARK_PATH))
# check length and keys to confirm json was imported correctly
print(len(hallmark_set))
print(list(hallmark_set.keys())[:10])

probe_to_gene = json.load(open(PROBE_TO_GENE_PATH))
# check length and keys to confirm json was imported correctly
print(len(probe_to_gene))
print(list(probe_to_gene.keys())[:10])

(59, 30065)
diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64
(5, 10)
Index(['Name', 'Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'FWER p-val',
       'Tag %', 'Gene %', 'Lead_genes'],
      dtype='str')
5
['HALLMARK_INFLAMMATORY_RESPONSE', 'HALLMARK_INTERFERON_ALPHA_RESPONSE', 'HALLMARK_INTERFERON_GAMMA_RESPONSE', 'HALLMARK_IL6_JAK_STAT3_SIGNALING', 'HALLMARK_COMPLEMENT']
54675
['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at', '1294_at', '1316_at', '1320_at', '1405_i_at', '1431_at']


`merged_df` returns the correct shape (59, 30065), and has the correct diagnosis distribution, confirming it was loaded correctly. `gsea_results` has the correct number of rows (5 for the five Hallmark sets) and the correct column count (10) and names. `hallmark_set` has returns the correct 5 keys. `probe_to_gene` has the correct length of 54675, matching it's length in `02_enrichment_analysis.ipynb`.

## 3.2 Feature Engineering

The current `merged_df` can't be used to train the classifier model since it currently has 30061 probe columns with only 58 training samples, p>>n, which risks overfitting. Additionally, in order to answer the second part of my primary hypothesis, I need to be able to compare the top features against the Hallmark leading-edge genes, but the Hallmark gene sets and `Lead_genes` column from GSEA are expressed in gene symbol not probe IDs. If I trained a model on the current expression matrix, it would prevent me from being able to directly compare top features against leading-edge genes. Therefore, in this section I will construct a samples x genes matrix that can be used to train a classifier model without risking overfitting, and in the correct format to be used to answer the hypothesis. 

### Gene Symbol x Probe Mean
To build the samples x genes matrix I first need to map the probes to the gene symbols, but since the probes have duplicates and are columns I can't just merge on the highest t-statistic like I did for enrichment analysis. Instead, I use the mean expression of the probes to collapse the duplicate probes. I drop the name columns from `merged_df` and take the mean of the each probe giving me a series of probes and their means. Then I map the series to `probe_to_gene` from enrichment analysis and save gene symbol and probe mean into a dataframe: `gene_and_probe_mean`. 

In [28]:
# probes to mean expression : chose mean since the small smaple size would make a variance approach unreliable
probe_means = merged_df.drop(columns=['sample_id', 'diagnosis', 'duration', 'name']).mean()
gene_symbol = probe_means.index.map(probe_to_gene)
data = {'gene symbol': gene_symbol, 'probe mean': probe_means}
gene_and_probe_mean =  pd.DataFrame(data)

**Sanity Checks**

Check shape and columns of the new df to ensure all rows and columns are included. 

In [30]:
print(gene_and_probe_mean.shape)
print(gene_and_probe_mean.columns)

(30061, 2)
Index(['gene symbol', 'probe mean'], dtype='str')


`.shape` matches the correct row and column count of (30061, 2). `.columns` returns the two correct columns, gene symbol and probe mean.

Check the number of missing values by getting the sum of `.isna()`.

In [31]:
gene_and_probe_mean['gene symbol'].isna().sum()

np.int64(4899)

There are 4899 missing values, which is expected and will be dropped later.

To spot check a specific gene symbol to ensure probe means are genuine and not noise, I first look at the top 10 repeated gene symbols, then take one of them and look at its probe mean in `gene_and_probe_mean`. 

In [32]:
gene_and_probe_mean['gene symbol'].value_counts().head(10)

gene symbol
MALAT1    12
ZBTB20    11
QKI       10
FGFR2     10
MSI2       9
MEG3       9
PPARA      9
DNAH1      9
HCG18      9
STRN       9
Name: count, dtype: int64

In [33]:
gene_and_probe_mean[gene_and_probe_mean['gene symbol'] == 'MALAT1']

,gene symbol,probe mean
1558678_s_at,MALAT1,12.173298
223577_x_at,MALAT1,8.695314
223578_x_at,MALAT1,7.568805
223940_x_at,MALAT1,9.859788
224558_s_at,MALAT1,9.165053
224559_at,MALAT1,8.886853
224567_x_at,MALAT1,10.395358
224568_x_at,MALAT1,9.341297
226675_s_at,MALAT1,8.959744
227510_x_at,MALAT1,9.042417


This shows there is meanigful difference in probe mean expression values, confirming they are genuinely distinct measurements and not noise. 

To drop the rows with missing gene symbols I use `.dropna(subset=['gene symbol'])`. To drop the duplicate gene symbols I use `probe_mean` to sort the values then drop the duplicates keeping the highest mean.

In [34]:
# drop row with missing gene symbols
gene_and_probe_mean = gene_and_probe_mean.dropna(subset=['gene symbol'])
print('shape of gene_and_probe_mean after drop missing', gene_and_probe_mean.shape)

# sort probes by probe mean and drop duplicates
gene_and_probe_mean = gene_and_probe_mean.sort_values('probe mean', ascending=False).drop_duplicates(subset='gene symbol', keep='first')
print('shape of gene_and_probe_mean after sort and drop duplicates', gene_and_probe_mean.shape)


shape of gene_and_probe_mean after drop missing (25162, 2)
shape of gene_and_probe_mean after sort and drop duplicates (15706, 2)


After dropping missing values the row count drops from 30061 to 25162 (30061 - 25162 = 4899) which is the correct number of missing values dropped. The final shape for `gene_and_probe_mean` after dropping duplicates is (15706, 2) which also matches the row count of the DE dataframe in `02_enrichment_analysis.ipynb`. 

To build the intermediate subsetted matrix for the final samples x genes matrix, I used `gene_and_probe_mean`'s index to subset `merged_df` in order to get a matrix that contains the metadata along with the non-duplicated probes so that the final samples x genes matrix contains the actual per-sample expression values that is stored in the `merged_df`.

In [37]:
probe_list = list(gene_and_probe_mean.index) 

# subset merged_df to only include non-duplicate probes and metadata
subset_merged_df = merged_df[['sample_id', 'diagnosis', 'duration', 'name'] + probe_list]

# check shape of new df
print(subset_merged_df.shape)

(59, 15710)


The shape (59, 15710) confirms the all 59 samples are included as rows and the 15710 columns are the 4 metadata rows + the gene columns.

To build the final samples x genes matrix I need to change `subset_merged_df`'s 15706 probe ID columns to their gene symbols. To do this I use `gene_and_probe_mean` to map the probe IDs to gene symbols. 

In [42]:
rename = gene_and_probe_mean['gene symbol'].to_dict()
samples_x_genes = subset_merged_df.rename(columns=rename)
print(samples_x_genes.shape)
print(samples_x_genes.columns[:10])
print((samples_x_genes['MALAT1'] == merged_df['1558678_s_at']).all())

(59, 15710)
Index(['sample_id', 'diagnosis', 'duration', 'name', 'SNAP25', 'CST3',
       'SLC1A2', 'MALAT1', 'KIF5A', 'STXBP1'],
      dtype='str')
True


Shape remains the same (59, 15710) confirming the correct number of samples and gene symbols. `.columns[:10]` shows column names are gene symbols not probe IDs mixed with metadata column names. Spot checking `MALAT1` values match `merged_df['1558678_s_at']`, which was the probe the duplicate `MALAT1` genes collapsed on, returns true veryifying renaming was done correctly. These checks confirm the `samples_x_genes` matrix was constructed correctly and is ready to be used for modeling.

## 3.3 Preprocessing Decisions



## 3.4 Modeling + Cross-Validation — Leave-one-out (LOO) 

## 3.5 Evaluation

## 3.6 Feature Importance

## 3.7 Summary